In [93]:
# pip freeze > requirements.txt (to save the current environment)

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datetime import datetime
# TODO: Model import goes here


# NBA API Endpoints
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.endpoints import boxscoresummaryv2
from nba_api.stats.endpoints import teamgamelog




## Workflow:
1. [⚠️FRONTEND] Get player name 
2. ✅ Get player ID from player name
3. Fetch career stats, gamelog, team gamelog using player ID
    Stats to fetch:
    - ✅ Career stats (averaged stats) [Need for predicting over/under prop]
    - Game log (past season games; grab prev season if <25% season completed; grab past 2 seasons games against certain team) [Need for dataset]
        - Core stats (PTS, REBOUNDS, ASSISTS, STEALS, 3PT MADE, DOUBLE-DOUBLES, TRIPLE-DOUBLES) [For dataset]
    - Box score summary (injury report) [Need for injury report]
    
4. Form dataset from all fetched endpoints
5. Preprocess the dataset
6. Train-test split (or any other splitting method)
7. Train the model + evaluate (fine tune)
8. Run predictions
9. [⚠️LLM] Get LLM to summarize the predictions


In [94]:
# List of all players in the NBA currently (will be used to autofill in search bar)
from nba_api.stats.static.players import get_players
players = pd.DataFrame(get_players())

# Function to get player ID from player name
def get_player_id(player_name):
    active_players = players[players['is_active']== True]

    # Retrieve player ID provided player name
    player_id = active_players[active_players['full_name'] == player_name]['id'].values[0]
    return player_id

test_name = 'Cade Cunningham' # TODO: Get user input from frontend (MUI autocomplete)

player_id = get_player_id(test_name) # Get player ID from player name
print(test_name, player_id) # Print player ID


Cade Cunningham 1630595


In [95]:
# Fetching avg career data
avg_career_stats = playercareerstats.PlayerCareerStats(player_id=player_id, per_mode36="PerGame").get_data_frames()[3] # Averaged career stats for regular season [Need for predicting prop]
print(avg_career_stats) # Print average career stats

   PLAYER_ID LEAGUE_ID  TEAM_ID  GP  GS   MIN  FGM   FGA  FG_PCT  FG3M  ...  \
0    1630595        00        0   6   6  41.2  9.2  21.5   0.426   0.8  ...   

   FT_PCT  OREB  DREB  REB  AST  STL  BLK  TOV   PF   PTS  
0   0.833   1.0   7.3  8.3  8.7  1.8  1.3  5.3  3.3  25.0  

[1 rows x 24 columns]


In [92]:
# Function to get current and previous NBA season based on the current date
def get_season():
    today = datetime.today()
    year = today.year
    month = today.month

    if month >= 10:  # October to December: new season starts
        start_year = year
    else:  # January to September: still part of the previous season
        start_year = year - 1

    current_season = f"{start_year}-{str(start_year + 1)[-2:]}"
    previous_season = f"{start_year - 1}-{str(start_year)[-2:]}"
    
    return current_season, previous_season

current_season, previous_season = get_season()

commoninfo = commonplayerinfo.CommonPlayerInfo(player_id=player_id) # Player info
team_id = commoninfo.get_data_frames()[0]['TEAM_ID'].values[0]

# Function to get player game logs
def get_player_gamelog(player_id, team_id):
    team_gamelog = teamgamelog.TeamGameLog(team_id=team_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Team game log
    
    # Check if team has played at least 21 games in the current season (1/4 season)
    if team_gamelog.count()['Game_ID'] <= 20:
        print("Not enough games played in current season, using previous season data as well")
        prev_gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=previous_season, season_type_all_star="Regular Season").get_data_frames()[0] # Previous season game log
        curr_gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Current season game log
        gamelog = pd.concat([curr_gamelog, prev_gamelog], ignore_index=True) # Combine current and previous season game logs

    else:
        print("Enough games played in current season, using current season data only")
        gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Game log

    return gamelog

print(get_player_gamelog(player_id=player_id, team_id=team_id)) # Print game log

# boxscore = boxscoresummaryv2.BoxScoreSummaryV2(player_id=player_id) # Box score summary [Need for injury report] (TODO: Verify parameters)





Enough games played in current season, using current season data only
   SEASON_ID  Player_ID     Game_ID     GAME_DATE      MATCHUP WL  MIN  FGM  \
0      22024    1630595  0022401171  Apr 11, 2025  DET vs. MIL  L   36   15   
1      22024    1630595  0022401167  Apr 10, 2025  DET vs. NYK  W   35   16   
2      22024    1630595  0022401144  Apr 07, 2025  DET vs. SAC  L   33   13   
3      22024    1630595  0022401129  Apr 05, 2025  DET vs. MEM  L   28    9   
4      22024    1630595  0022401019  Mar 21, 2025    DET @ DAL  L   38   15   
..       ...        ...         ...           ...          ... ..  ...  ...   
65     22024    1630595  0022400120  Oct 30, 2024    DET @ PHI  W   37    6   
66     22024    1630595  0022400105  Oct 28, 2024    DET @ MIA  L   35    9   
67     22024    1630595  0022400089  Oct 26, 2024  DET vs. BOS  L   41    9   
68     22024    1630595  0022400080  Oct 25, 2024    DET @ CLE  L   35   14   
69     22024    1630595  0022400063  Oct 23, 2024  DET vs. IN